In [2]:
# Imports

from pathlib import Path
import subprocess

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
REPO_URL = "https://github.com/LunusMax/epl-seriea-data.git"
DATA_DIR = Path("data/epl-seriea-data")

def download_dataset():
    if DATA_DIR.exists():
        print("Dataset already exists.")
        return

    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)

    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(DATA_DIR)],
        check=True
    )

    print("Dataset downloaded successfully.")

download_dataset()

Dataset downloaded successfully.


In [3]:
# List all CSV files in the downloaded dataset
csv_files = sorted(DATA_DIR.rglob("*.csv"))

print(f"Total CSV files: {len(csv_files)}")

for file in csv_files[:20]:
    print(file.relative_to(DATA_DIR))

Total CSV files: 277
Games\premier_league_2018-2019_resultados.csv
Games\premier_league_2019-2020_resultados.csv
Games\premier_league_2020-2021_resultados.csv
Games\premier_league_2021-2022_resultados.csv
Games\premier_league_2022-2023_resultados.csv
Games\premier_league_2023-2024_resultados.csv
Games\serie_a_2018-2019_resultados.csv
Games\serie_a_2019-2020_resultados.csv
Games\serie_a_2020-2021_resultados.csv
Games\serie_a_2021-2022_resultados.csv
Games\serie_a_2022-2023_resultados.csv
Games\serie_a_2023-2024_resultados.csv
Premier_League\data_arsenal_2018-2019.csv
Premier_League\data_arsenal_2019-2020.csv
Premier_League\data_arsenal_2020-2021.csv
Premier_League\data_arsenal_2021-2022.csv
Premier_League\data_arsenal_2022-2023.csv
Premier_League\data_arsenal_2023-2024.csv
Premier_League\data_aston-villa_2019-2020.csv
Premier_League\data_aston-villa_2020-2021.csv


In [4]:
# ============================================================
# Player dataset inspection
# ============================================================

player_files = [
    file for file in csv_files
    if file.parent.name in ["Premier_League", "Serie_A"]
]

games_files = [
    file for file in csv_files
    if file.parent.name == "Games"
]

print(f"Player CSV files: {len(player_files)}")
print(f"Games CSV files: {len(games_files)}")

for league in ["Premier_League", "Serie_A"]:
    league_files = [
        file for file in player_files
        if file.parent.name == league
    ]

    print(f"\n{league}: {len(league_files)} files")

# Inspect one player CSV
sample_file = player_files[0]
sample_df = pd.read_csv(sample_file)

print(f"\nSample file: {sample_file.name}")
print(f"Shape: {sample_df.shape}")

print("\nColumns:")
print(sample_df.columns.tolist())

display(sample_df.head())

Player CSV files: 265
Games CSV files: 12

Premier_League: 120 files

Serie_A: 145 files

Sample file: data_arsenal_2018-2019.csv
Shape: (31, 48)

Columns:
['Jogador', 'Pos.', 'Idade', 'MP', 'Inícios', 'Min.', '90s', 'Gols', 'Assis.', 'G+A', 'G-PB', 'PB', 'PT', 'CrtsA', 'CrtV', 'xG', 'npxG', 'xAG', 'npxG+xAG', 'PrgC', 'PrgP', 'PrgR', 'Gols.1', 'Assis..1', 'G+A.1', 'G-PB.1', 'G+A-PB', 'xG.1', 'xAG.1', 'xG+xAG', 'npxG.1', 'npxG+xAG.1', 'Div', 'TklW', 'Terço Def', 'Terço Central', 'Terço de Ataque', 'Div.1', 'Tent', 'Tkl%', 'Perdido', 'Bloqueios', 'TC', 'Passe', 'Crts', 'Tkl+Int', 'Def', 'Erros']


,Jogador,Pos.,Idade,MP,Inícios,Min.,90s,Gols,Assis.,G+A,...,Tent,Tkl%,Perdido,Bloqueios,TC,Passe,Crts,Tkl+Int,Def,Erros
0,Bernd Leno,G,26,32,31,2835.0,31.5,0,0,0,...,3,0.0,3,0,0,0,4,4,16,6
1,Shkodran Mustafi,ZG,26,31,31,2610.0,29.0,2,0,2,...,40,77.5,9,34,17,17,58,120,168,1
2,Pierre-Emerick Aubameyang,AT,29,36,30,2727.0,30.3,22,5,27,...,28,28.6,20,26,0,26,8,29,22,0
3,Granit Xhaka,"LT,ZG",25,29,29,2502.0,27.8,4,2,6,...,59,40.7,35,35,9,26,36,88,56,3
4,Alexandre Lacazette,AT,27,35,27,2502.0,27.8,13,8,21,...,28,17.9,23,27,1,26,15,50,31,0


In [5]:
# ============================================================
# Consolidate player datasets
# ============================================================

import re

dataframes = []
reference_columns = None
schema_differences = []

for file in player_files:
    df = pd.read_csv(file)

    # Check column consistency
    if reference_columns is None:
        reference_columns = list(df.columns)
    elif list(df.columns) != reference_columns:
        schema_differences.append(file.name)

    # Extract team and season from filename
    match = re.fullmatch(
        r"data_(.+)_(\d{4}-\d{4})",
        file.stem
    )

    if not match:
        raise ValueError(f"Unexpected filename: {file.name}")

    team, season = match.groups()

    # Add metadata
    df["Liga"] = file.parent.name
    df["Time"] = team
    df["Temporada"] = season
    df["Arquivo"] = file.name

    dataframes.append(df)

# Consolidate datasets
players_df = pd.concat(dataframes, ignore_index=True)

print(f"Total records: {len(players_df):,}")
print(f"Total columns (including metadata): {len(players_df.columns)}")
print(f"Files with different schemas: {len(schema_differences)}")

if schema_differences:
    print("\nFiles with different schemas:")
    print(schema_differences[:20])

print("\nRecords by league and season:")

season_summary = (
    players_df
    .groupby(["Liga", "Temporada"])
    .agg(
        Registros=("Jogador", "size"),
        Equipes=("Time", "nunique")
    )
    .reset_index()
    .sort_values(["Liga", "Temporada"])
)

display(season_summary)

print("\nDataset preview:")
display(players_df.head())


Total records: 8,931
Total columns (including metadata): 52
Files with different schemas: 0

Records by league and season:


,Liga,Temporada,Registros,Equipes
0,Premier_League,2018-2019,575,20
1,Premier_League,2019-2020,525,20
2,Premier_League,2020-2021,538,20
3,Premier_League,2021-2022,552,20
4,Premier_League,2022-2023,568,20
5,Premier_League,2023-2024,579,20
6,Serie_A,2018-2019,712,20
7,Serie_A,2019-2020,857,23
8,Serie_A,2020-2021,1075,27
9,Serie_A,2021-2022,1005,25



Dataset preview:


,Jogador,Pos.,Idade,MP,Inícios,Min.,90s,Gols,Assis.,G+A,...,TC,Passe,Crts,Tkl+Int,Def,Erros,Liga,Time,Temporada,Arquivo
0,Bernd Leno,G,26.0,32,31,2835.0,31.5,0.0,0.0,0.0,...,0.0,0.0,4.0,4.0,16.0,6.0,Premier_League,arsenal,2018-2019,data_arsenal_2018-2019.csv
1,Shkodran Mustafi,ZG,26.0,31,31,2610.0,29.0,2.0,0.0,2.0,...,17.0,17.0,58.0,120.0,168.0,1.0,Premier_League,arsenal,2018-2019,data_arsenal_2018-2019.csv
2,Pierre-Emerick Aubameyang,AT,29.0,36,30,2727.0,30.3,22.0,5.0,27.0,...,0.0,26.0,8.0,29.0,22.0,0.0,Premier_League,arsenal,2018-2019,data_arsenal_2018-2019.csv
3,Granit Xhaka,"LT,ZG",25.0,29,29,2502.0,27.8,4.0,2.0,6.0,...,9.0,26.0,36.0,88.0,56.0,3.0,Premier_League,arsenal,2018-2019,data_arsenal_2018-2019.csv
4,Alexandre Lacazette,AT,27.0,35,27,2502.0,27.8,13.0,8.0,21.0,...,1.0,26.0,15.0,50.0,31.0,0.0,Premier_League,arsenal,2018-2019,data_arsenal_2018-2019.csv


In [6]:
# Compare complete and processed datasets

print(f"Complete dataset: {len(players_df):,} records")

analysis_df = players_df[
    players_df["Temporada"].isin([
        "2019-2020",
        "2020-2021",
        "2021-2022",
        "2022-2023",
        "2023-2024"
    ])
].copy()

print(f"Research period: {len(analysis_df):,} records")

print("\nRecords by league:")
display(
    analysis_df.groupby("Liga")
    .agg(
        Registros=("Jogador", "size"),
        Equipes=("Time", "nunique")
    )
)

Complete dataset: 8,931 records
Research period: 7,644 records

Records by league:


,Registros,Equipes
Liga,,
Premier_League,2762,26
Serie_A,4882,29


In [7]:
# ============================================================
# Squad size analysis
# ============================================================

squad_sizes = (
    analysis_df
    .groupby(["Liga", "Temporada", "Time"])
    .size()
    .reset_index(name="Jogadores")
)

season_summary = (
    squad_sizes
    .groupby(["Liga", "Temporada"])
    .agg(
        Equipes=("Time", "nunique"),
        Registros=("Jogadores", "sum"),
        Media_Jogadores=("Jogadores", "mean"),
        Min_Jogadores=("Jogadores", "min"),
        Max_Jogadores=("Jogadores", "max")
    )
    .round(2)
    .reset_index()
    .sort_values(["Liga", "Temporada"])
)

display(season_summary)

print(f"Total club-season instances: {len(squad_sizes)}")

print(
    "Squads with fewer than 20 players:",
    (squad_sizes["Jogadores"] < 20).sum()
)

display(
    squad_sizes[
        squad_sizes["Jogadores"] < 20
    ].sort_values("Jogadores")
)


,Liga,Temporada,Equipes,Registros,Media_Jogadores,Min_Jogadores,Max_Jogadores
0,Premier_League,2019-2020,20,525,26.25,21,33
1,Premier_League,2020-2021,20,538,26.90,23,37
2,Premier_League,2021-2022,20,552,27.60,23,34
3,Premier_League,2022-2023,20,568,28.40,24,36
4,Premier_League,2023-2024,20,579,28.95,24,35
5,Serie_A,2019-2020,23,857,37.26,32,45
6,Serie_A,2020-2021,27,1075,39.81,32,51
7,Serie_A,2021-2022,25,1005,40.20,31,50
8,Serie_A,2022-2023,25,961,38.44,30,47
9,Serie_A,2023-2024,25,984,39.36,31,54


Total club-season instances: 225
Squads with fewer than 20 players: 0


,Liga,Temporada,Time,Jogadores


In [8]:
# ============================================================
# Serie A dataset consistency analysis
# ============================================================

serie_a_df = analysis_df[
    analysis_df["Liga"] == "Serie_A"
].copy()

# List teams by season
for season in sorted(serie_a_df["Temporada"].unique()):
    teams = sorted(
        serie_a_df.loc[
            serie_a_df["Temporada"] == season,
            "Time"
        ].unique()
    )

    print(f"\nSeason {season} - {len(teams)} teams")
    print(", ".join(teams))

# Check duplicated players within each squad
duplicates = serie_a_df[
    serie_a_df.duplicated(
        subset=["Temporada", "Time", "Jogador"],
        keep=False
    )
].sort_values(["Temporada", "Time", "Jogador"])

print("\nDuplicate player records within the same squad:")
print(len(duplicates))

if not duplicates.empty:
    display(
        duplicates[
            ["Temporada", "Time", "Jogador", "Arquivo"]
        ].head(30)
    )



Season 2019-2020 - 23 teams
atalanta, bologna, brescia, cagliari, cremonese, empoli, fiorentina, frosinone, genoa, hellas-verona, internazionale, juventus, lazio, lecce, milan, napoli, parma, roma, sampdoria, sassuolo, spal, torino, udinese

Season 2020-2021 - 27 teams
atalanta, benevento, bologna, cagliari, cremonese, crotone, empoli, fiorentina, frosinone, genoa, hellas-verona, internazionale, juventus, lazio, lecce, milan, monza, napoli, parma, roma, salernitana, sampdoria, sassuolo, spezia, torino, udinese, venezia

Season 2021-2022 - 25 teams
atalanta, bologna, cagliari, cremonese, empoli, fiorentina, frosinone, genoa, hellas-verona, internazionale, juventus, lazio, lecce, milan, monza, napoli, parma, roma, salernitana, sampdoria, sassuolo, spezia, torino, udinese, venezia

Season 2022-2023 - 25 teams
atalanta, bologna, cagliari, cremonese, empoli, fiorentina, frosinone, genoa, hellas-verona, internazionale, juventus, lazio, lecce, milan, monza, napoli, parma, roma, salernitana, 

## Identificação e Separação das Equipes por Temporada

In [9]:
# ============================================================
# Inspect potentially inconsistent Serie A files
# ============================================================

suspect_teams = [
    "cremonese",
    "monza",
    "frosinone",
    "empoli"
]

for team in suspect_teams:
    team_df = serie_a_df[
        serie_a_df["Time"] == team
    ]

    print(f"\n{'=' * 60}")
    print(f"TEAM: {team.upper()}")
    print(f"{'=' * 60}")

    summary = (
        team_df.groupby("Temporada")
        .agg(
            Registros=("Jogador", "size"),
            Jogadores_Unicos=("Jogador", "nunique")
        )
        .reset_index()
        .sort_values("Temporada")
    )

    display(summary)

    # Compare player names across seasons
    player_sets = {
        season: set(group["Jogador"].dropna())
        for season, group in team_df.groupby("Temporada")
    }

    seasons = sorted(player_sets)

    comparisons = []

    for i in range(len(seasons) - 1):
        season_a = seasons[i]
        season_b = seasons[i + 1]

        players_a = player_sets[season_a]
        players_b = player_sets[season_b]

        intersection = len(players_a & players_b)
        union = len(players_a | players_b)

        comparisons.append({
            "Season A": season_a,
            "Season B": season_b,
            "Common players": intersection,
            "Jaccard similarity": round(
                intersection / union, 3
            ) if union else 0
        })

    display(pd.DataFrame(comparisons))


TEAM: CREMONESE


,Temporada,Registros,Jogadores_Unicos
0,2019-2020,36,36
1,2020-2021,36,36
2,2021-2022,37,37
3,2022-2023,37,37
4,2023-2024,38,38


,Season A,Season B,Common players,Jaccard similarity
0,2019-2020,2020-2021,18,0.333
1,2020-2021,2021-2022,18,0.327
2,2021-2022,2022-2023,13,0.213
3,2022-2023,2023-2024,18,0.316



TEAM: MONZA


,Temporada,Registros,Jogadores_Unicos
0,2020-2021,44,44
1,2021-2022,35,35
2,2022-2023,36,36
3,2023-2024,39,39


...

,Temporada,Registros,Jogadores_Unicos
0,2019-2020,34,34
1,2020-2021,40,40
2,2021-2022,41,41
3,2022-2023,36,36
4,2023-2024,41,41


,Season A,Season B,Common players,Jaccard similarity
0,2019-2020,2020-2021,23,0.451
1,2020-2021,2021-2022,13,0.191
2,2021-2022,2022-2023,12,0.185
3,2022-2023,2023-2024,13,0.203



TEAM: EMPOLI


,Temporada,Registros,Jogadores_Unicos
0,2019-2020,44,44
1,2020-2021,36,36
2,2021-2022,40,40
3,2022-2023,43,43
4,2023-2024,42,42


,Season A,Season B,Common players,Jaccard similarity
0,2019-2020,2020-2021,21,0.356
1,2020-2021,2021-2022,19,0.333
2,2021-2022,2022-2023,18,0.277
3,2022-2023,2023-2024,22,0.349


In [10]:
# ============================================================
# Inspect match results datasets
# ============================================================

for file in games_files:
    df = pd.read_csv(file)

    print(f"\n{'=' * 60}")
    print(f"File: {file.name}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")

    display(df.head(3))
    
    # Inspect only one file from each league
    if "premier_league" in file.name:
        break

serie_a_game = next(
    file for file in games_files
    if "serie_a_2023-2024" in file.name
)

df = pd.read_csv(serie_a_game)

print(f"\n{'=' * 60}")
print(f"File: {serie_a_game.name}")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

display(df.head(3))



File: premier_league_2018-2019_resultados.csv
Shape: (380, 3)
Columns: ['Em casa', 'Visitante', 'Classe']


,Em casa,Visitante,Classe
0,Manchester Utd,Leicester City,1
1,Newcastle Utd,Tottenham,2
2,Fulham,Crystal Palace,2



File: serie_a_2023-2024_resultados.csv
Shape: (380, 3)
Columns: ['Em casa', 'Visitante', 'Classe']


,Em casa,Visitante,Classe
0,Frosinone,Napoli,2.0
1,Empoli,Hellas Verona,2.0
2,Inter,Monza,1.0


In [11]:
# ============================================================
# Compare Serie A teams between player and match datasets
# ============================================================

import unicodedata
import re

def normalize_team_name(name):
    name = str(name).lower().strip()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r"[^a-z0-9]+", " ", name).strip()

    aliases = {
        "internazionale": "inter",
        "inter milan": "inter",
        "ac milan": "milan",
        "as roma": "roma",
        "hellas verona": "hellas verona",
    }

    return aliases.get(name, name)


comparison_results = []

for season in sorted(serie_a_df["Temporada"].unique()):

    game_file = (
        DATA_DIR / "Games" / f"serie_a_{season}_resultados.csv"
    )

    games_df = pd.read_csv(game_file)

    # Teams participating in the competition
    official_teams = set(
        games_df["Em casa"].dropna().map(normalize_team_name)
    ) | set(
        games_df["Visitante"].dropna().map(normalize_team_name)
    )

    # Teams represented in player CSV files
    dataset_teams = set(
        serie_a_df.loc[
            serie_a_df["Temporada"] == season, "Time"
        ].map(normalize_team_name)
    )

    matched = dataset_teams & official_teams
    extra = dataset_teams - official_teams
    missing = official_teams - dataset_teams

    comparison_results.append({
        "Temporada": season,
        "Equipes_Jogos": len(official_teams),
        "Equipes_Dados": len(dataset_teams),
        "Correspondentes": len(matched),
        "Extras": len(extra),
        "Ausentes": len(missing),
    })

    print(f"\n{'=' * 55}")
    print(f"Season: {season}")
    print(f"Teams in match results: {len(official_teams)}")
    print(f"Teams in player data: {len(dataset_teams)}")
    print(f"Matched teams: {len(matched)}")
    print(f"Extra teams: {sorted(extra)}")
    print(f"Missing teams: {sorted(missing)}")

comparison_df = pd.DataFrame(comparison_results)

print("\nSummary:")
display(comparison_df)



Season: 2019-2020
Teams in match results: 20
Teams in player data: 23
Matched teams: 20
Extra teams: ['cremonese', 'empoli', 'frosinone']
Missing teams: []

Season: 2020-2021
Teams in match results: 20
Teams in player data: 27
Matched teams: 20
Extra teams: ['cremonese', 'empoli', 'frosinone', 'lecce', 'monza', 'salernitana', 'venezia']
Missing teams: []

Season: 2021-2022
Teams in match results: 20
Teams in player data: 25
Matched teams: 20
Extra teams: ['cremonese', 'frosinone', 'lecce', 'monza', 'parma']
Missing teams: []

Season: 2022-2023
Teams in match results: 20
Teams in player data: 25
Matched teams: 20
Extra teams: ['cagliari', 'frosinone', 'genoa', 'parma', 'venezia']
Missing teams: []

Season: 2023-2024
Teams in match results: 20
Teams in player data: 25
Matched teams: 20
Extra teams: ['cremonese', 'parma', 'sampdoria', 'spezia', 'venezia']
Missing teams: []

Summary:


,Temporada,Equipes_Jogos,Equipes_Dados,Correspondentes,Extras,Ausentes
0,2019-2020,20,23,20,3,0
1,2020-2021,20,27,20,7,0
2,2021-2022,20,25,20,5,0
3,2022-2023,20,25,20,5,0
4,2023-2024,20,25,20,5,0


In [12]:
# ============================================================
# Filter Serie A teams by official competition participation
# ============================================================

official_teams_by_season = {}

for season in sorted(serie_a_df["Temporada"].unique()):
    game_file = DATA_DIR / "Games" / f"serie_a_{season}_resultados.csv"
    games_df = pd.read_csv(game_file)

    official_teams = set(
        games_df["Em casa"].dropna().map(normalize_team_name)
    ) | set(
        games_df["Visitante"].dropna().map(normalize_team_name)
    )

    official_teams_by_season[season] = official_teams

# Identify records belonging to participating teams
official_mask = serie_a_df.apply(
    lambda row: normalize_team_name(row["Time"])
    in official_teams_by_season[row["Temporada"]],
    axis=1
)

# Keep both datasets
serie_a_official_df = serie_a_df.loc[official_mask].copy()
serie_a_extra_df = serie_a_df.loc[~official_mask].copy()

print(f"Original records: {len(serie_a_df):,}")
print(f"Official Serie A records: {len(serie_a_official_df):,}")
print(f"Additional records: {len(serie_a_extra_df):,}")

print("\nOfficial teams by season:")

official_summary = (
    serie_a_official_df
    .groupby("Temporada")
    .agg(
        Registros=("Jogador", "size"),
        Equipes=("Time", "nunique")
    )
    .reset_index()
)

display(official_summary)

print("\nAdditional records by season:")

extra_summary = (
    serie_a_extra_df
    .groupby("Temporada")
    .agg(
        Registros=("Jogador", "size"),
        Equipes=("Time", "nunique")
    )
    .reset_index()
)

display(extra_summary)

# Validate filtering
assert len(serie_a_official_df) + len(serie_a_extra_df) == len(serie_a_df)
assert (official_summary["Equipes"] == 20).all()
assert official_summary["Equipes"].sum() == 100

print("\nValidation successful.")


Original records: 4,882
Official Serie A records: 3,899
Additional records: 983

Official teams by season:


,Temporada,Registros,Equipes
0,2019-2020,743,20
1,2020-2021,800,20
2,2021-2022,808,20
3,2022-2023,761,20
4,2023-2024,787,20



Additional records by season:


,Temporada,Registros,Equipes
0,2019-2020,114,3
1,2020-2021,275,7
2,2021-2022,197,5
3,2022-2023,200,5
4,2023-2024,197,5



Validation successful.


## Análise da Qualidade dos Dados